# Create Kyoto Prize Awards (PRIZE PATTERN)

Kyoto Prize laureates from the official Kyoto Prize website published by the Inamori Foundation.

**Prerequisite:** run `scripts/local/kyoto_prize_to_s3.py` to fetch the official laureates index, official detail pages, verify the official current amount-rule page, and upload parquet to S3.

**Source:** `https://www.kyotoprize.org/en/laureates/` plus official laureate detail pages, `https://www.kyotoprize.org/en/about/` for the current amount rule, and `https://www.kyotoprize.org/wp-content/uploads/2019/08/rita_everlasting_en.pdf` for the 2018 increase statement.

**S3 location:** `s3://openalex-ingest/awards/kyoto_prize/kyoto_prize_laureates.parquet`

**OpenAlex funder:** Inamori Foundation, `funder_id = 4320322210`. OpenAlex did not expose a DOI/ROR in public Step 0 checks.

**Schema notes:**
- Prize pattern: one row per official laureate record.
- `lead_investigator` is the laureate.
- `funding_type = 'prize'`.
- Provenance is `kyoto_prize`.
- Official current prize money is 100,000,000 JPY per category. The 2018 official statement supports the increase timing, so 2018-present amounts are populated and apportioned across shared year/field laureates. Pre-2018 amount/currency values are NULL.
- The source includes one organization laureate, The Nobel Foundation; the raw `is_organization_laureate` flag documents that case, and the transform preserves it in `lead_investigator.family_name`.
- Affiliation text is mapped to `lead_investigator.affiliation.name` when the official detail page publishes it. Country and external affiliation IDs are NULL because the source does not provide structured institution-country identifiers.

**Contractor handoff:** local validation was completed without AWS/S3 or Databricks credentials. Admin must upload the parquet, run this notebook, run QA, and then run `CreateAwards.ipynb`. No job YAML is included until admin verification.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kyoto_prize_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kyoto_prize/kyoto_prize_laureates.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.kyoto_prize_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.


In [ ]:
%sql
DESCRIBE openalex.awards.kyoto_prize_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.kyoto_prize_raw
LIMIT 10;


In [ ]:
%sql
-- Money-shaped columns from the raw source. Amounts are derived in the script
-- from official Kyoto Prize amount-rule material and stored as strings.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'kyoto_prize_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|fund|award|value|money|prize|yen|jpy';


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(source_award_amount) AS rows_with_amount,
    MIN(TRY_CAST(source_award_amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(source_award_amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(source_award_amount AS DOUBLE)) AS avg_amount,
    collect_set(amount_rule_url) AS amount_rule_urls,
    collect_set(amount_rule_note) AS amount_rule_notes
FROM openalex.awards.kyoto_prize_raw
GROUP BY currency
ORDER BY currency;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(laureate_name) AS has_name,
    COUNT(award_year) AS has_year,
    COUNT(prize_category) AS has_category,
    COUNT(prize_field) AS has_field,
    COUNT(affiliation) AS has_affiliation,
    COUNT(citation) AS has_citation,
    COUNT(source_award_amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.kyoto_prize_raw;


In [ ]:
%sql
SELECT
    award_year,
    prize_category,
    prize_field,
    laureate_name,
    laureate_role,
    affiliation,
    source_award_amount,
    currency,
    landing_page_url
FROM openalex.awards.kyoto_prize_raw
ORDER BY TRY_CAST(award_year AS INT) DESC, prize_category, prize_field, laureate_name
LIMIT 50;


In [ ]:
%sql
SELECT
    award_year,
    prize_field,
    COUNT(*) AS laureate_rows,
    COUNT(source_award_amount) AS rows_with_amount,
    SUM(TRY_CAST(source_award_amount AS DOUBLE)) AS total_amount
FROM openalex.awards.kyoto_prize_raw
GROUP BY award_year, prize_field
ORDER BY TRY_CAST(award_year AS INT) DESC, prize_field;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the `CROSS JOIN` would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320322210;


## Step 2: Transform to OpenAlex Awards Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kyoto_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322210
),
normalized AS (
    SELECT
        n.*,
        TRY_CAST(n.award_year AS INT) AS award_year_int,
        TRY_CAST(n.source_award_amount AS DOUBLE) AS amount_double
    FROM openalex.awards.kyoto_prize_raw n
),
awards AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':kyoto-prize:', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        CONCAT('Kyoto Prize ', TRY_CAST(n.award_year_int AS STRING), ' - ', n.prize_field, ' - ', n.laureate_name) AS display_name,
        COALESCE(
            NULLIF(n.citation, ''),
            NULLIF(n.achievement_digest, ''),
            NULLIF(n.profile_description, ''),
            NULLIF(n.achievement_title, '')
        ) AS description,
        f.funder_id,
        n.funder_award_id,
        n.amount_double AS amount,
        NULLIF(n.currency, '') AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'prize' AS funding_type,
        COALESCE(NULLIF(n.prize_field, ''), NULLIF(n.prize_category, '')) AS funder_scheme,
        'kyoto_prize' AS provenance,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            NULLIF(n.given_name, '') AS given_name,
            NULLIF(n.family_name, '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                NULLIF(n.affiliation, '') AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        n.landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.laureate_name IS NOT NULL
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G', TRY_CAST(id AS STRING)) AS works_api_url,
    created_date,
    updated_date
FROM awards;


## Step 3: Insert Into `openalex_awards_raw` With Priority 70


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kyoto_prize' AND priority = 70;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    70 AS priority
FROM openalex.awards.kyoto_prize_awards;


## Verification


In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.kyoto_prize_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.kyoto_prize_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.kyoto_prize_awards;


In [ ]:
%sql
SELECT
    funder.display_name AS funder_display_name,
    funder_id,
    COUNT(*) AS rows
FROM openalex.awards.kyoto_prize_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT
    funder_scheme,
    currency,
    COUNT(*) AS rows,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.kyoto_prize_awards
GROUP BY funder_scheme, currency
ORDER BY funder_scheme, currency;


In [ ]:
%sql
SELECT
    start_year,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount
FROM openalex.awards.kyoto_prize_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Duplicate checks. Both should return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.kyoto_prize_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.kyoto_prize_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    amount,
    currency,
    funder_scheme,
    start_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.kyoto_prize_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 20;
